## TZ ref admixture - analysis

In [1]:
!pip install -q malariagen_data bed_reader
#typing-extensions<4.6.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 40.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 114.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 134.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.5/302.5 kB 34.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.7/138.7 kB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 91.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 120.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.9/206.9 kB 23.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 94.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.8 MB/s e

In [2]:
!pip install -q malariagen_data

  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user


In [1]:
!pip install malariagen_data "xyzservices < 2023.10.1"

  Using cached malariagen_data-7.14.1-py3-none-any.whl (132 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 1.8 MB/s eta 0:00:00
  Using cached biopython-1.81-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Using cached dash-2.14.1-py3-none-any.whl (10.4 MB)
  Using cached dash_cytoscape-0.3.0-py3-none-any.whl (3.6 MB)
  Using cached igv_notebook-0.5.2-py3-none-any.whl (302 kB)
  Using cached importlib_metadata-4.13.0-py3-none-any.whl (23 kB)
  Using cached ipinfo-4.4.3-py3-none-any.whl (24 kB)
  Using cached jupyter_dash-0.4.2-py3-none-any.whl (23 kB)
  Using cached numpydoc_decorator-2.1.0-py3-none-any.whl (11 kB)
  Using cached orjson-3.9.10-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (138 kB)
  Using cached protopunica-0.14.8-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (20.9 MB)
  Using cached scikit_allel-1.3.7-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.1 MB)
  Using cached typeguard-4.1.5-py3-no

In [2]:
!pip install bed_reader

  Using cached bed_reader-1.0.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.1 MB)


In [3]:
import malariagen_data
import bed_reader
import dask.array as da
import allel
import numpy as np
import plotly.express as px
import pandas as pd
from dask.diagnostics import ProgressBar
import google.colab
import os

In [4]:
google.colab.drive.mount("drive")

Mounted at drive


In [5]:
results_dir = "drive/MyDrive/Tanzania/TZ-ref-admixture-131123"
!mkdir -pv "{results_dir}"

mkdir: created directory 'drive/MyDrive/Tanzania/TZ-ref-admixture-131123'


In [6]:
# do this just to check client location
malariagen_data.Ag3()

<MalariaGEN Ag3 API client>
Storage URL             : gs://vo_agam_release/
Data releases available : 3.0
Results cache           : None
Cohorts analysis        : 20231019
AIM analysis            : 20220528
Site filters analysis   : dt_20200416
Software version        : malariagen_data 7.14.1
Client location         : Iowa, US
---
Please note that data are subject to terms of use,
for more information see https://www.malariagen.net/data
or contact data@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v7.14.1/Ag3.html

In [7]:
ag3 = malariagen_data.Ag3()

In [8]:
def get_plink_file_path(
    contig,
    n_snps,
    min_minor_ac,
    thin_offset,
    max_missing_an,
):
    return f"{results_dir}/{contig}.{n_snps}.{min_minor_ac}.{thin_offset}.{max_missing_an}"


In [9]:
def snp_calls_to_plink(
    contig,
    n_snps,
    min_minor_ac,
    thin_offset,
    max_missing_an=0,
    sample_sets=None,
    sample_query=None,
    site_mask = "gamb_colu_arab",
    #min_cohort_size=10,
    #max_cohort_size=50,
):
    contig_label = contig.replace(",", ".")
    contig_label = f"chr{contig_label}"
    plink_file_path = get_plink_file_path(
        contig=contig_label,
        n_snps=n_snps,
        thin_offset=thin_offset,
        min_minor_ac=min_minor_ac,
        max_missing_an=max_missing_an,
    )
    bed_file_path = f"{plink_file_path}.bed"
    if os.path.exists(bed_file_path):
        return plink_file_path

    print(f">>> building {plink_file_path}")

    print("access SNP calls")
    ds_snps = ag3.snp_calls(
        region=contig,
        sample_sets = sample_sets,
        sample_query = sample_query,
        site_mask=site_mask,
        #min_cohort_size=min_cohort_size, ##??????? shouldn't this deal with cohort queries seperately? use setup_cohorts?
        #max_cohort_size=max_cohort_size,
    )

    #####
    #selection of samples by taxon. Maximum of 50.
    #df_samples = ag3.sample_metadata(sample_query=sample_query)
    #index_by_taxon = df_samples.reset_index().groupby('taxon').index.unique().reset_index()

    #loc_index = []
    #for index_list in range (len(index_by_taxon.taxon)):
    #  if (len(index_by_taxon['index'][index_list])) > 50:
    #    selection = np.random.choice(index_by_taxon['index'][index_list], 50, replace=False)
    #    loc_index.append(selection)
    #  else:
    #    loc_index.append(index_by_taxon['index'][index_list])
    #flatten list
    #loc_index = [item for sublist in loc_index for item in sublist]
    #ds_snps_sel = ds_snps.sel(samples=loc_index)

    print("count alleles")
    gt = allel.GenotypeDaskArray(ds_snps["call_genotype"].data)
    #gt = allel.GenotypeDaskArray(ds_snps_sel["call_genotype"].data)
    with ProgressBar():
        ac = gt.count_alleles(max_allele=3).compute()

    print("ascertain segregating biallelic sites")
    n_chroms = ds_snps.dims["samples"] * 2
    #n_chroms = ds_snps_sel.dims["samples"] * 2
    an_called = ac.sum(axis=1)
    an_missing = n_chroms - an_called
    min_ref_ac = min_minor_ac
    max_ref_ac = n_chroms - min_minor_ac
    # here we choose biallelic sites involving the reference allele
    loc_sites = (
        ac.is_biallelic()
        & (ac[:, 0] >= min_ref_ac)
        & (ac[:, 0] <= max_ref_ac)
        & (an_missing <= max_missing_an)
    )
    print(f"ascertained {np.count_nonzero(loc_sites):,} sites")

    print("thin sites")
    ix_sites = np.nonzero(loc_sites)[0]
    thin_step = max(ix_sites.shape[0] // n_snps, 1)
    ix_sites_thinned = ix_sites[thin_offset::thin_step]
    print(f"thinned to {np.count_nonzero(ix_sites_thinned):,} sites")

    print("set up dataset")
    ds_snps_asc = (
        ds_snps
        #ds_snps_sel
        [["variant_contig", "variant_position", "variant_allele", "sample_id", "call_genotype"]]
        .isel(alleles=slice(0, 2))
        .sel(variants=ix_sites_thinned)
    )

    print("compute genotype ref counts")
    gt = ds_snps_asc["call_genotype"].data
    gn_ref = allel.GenotypeDaskArray(gt).to_n_ref(fill=-127)
    with ProgressBar():
        gn_ref = gn_ref.compute()

    print("ensure genotypes vary")
    loc_var = np.any(gn_ref != gn_ref[:, 0, np.newaxis], axis=1)
    print(f"final no. variants {np.count_nonzero(loc_var)}")

    print("load final data")
    with ProgressBar():
        ds_snps_final = (
            ds_snps_asc
            [["variant_contig", "variant_position", "variant_allele", "sample_id"]]
            .isel(variants=loc_var)
        )
    gn_ref_final = gn_ref[loc_var]
    val = gn_ref_final.T
    alleles = ds_snps_final["variant_allele"].values
    properties = {
        "iid": ds_snps_final["sample_id"].values,
        "chromosome": ds_snps_final["variant_contig"].values,
        "bp_position": ds_snps_final["variant_position"].values,
        "allele_1": alleles[:, 0],
        "allele_2": alleles[:, 1],
    }

    print("write plink files")
    bed_reader.to_bed(
        filepath=bed_file_path,
        val=val,
        properties=properties,
        count_A1=True,
    )
    return plink_file_path


In [ ]:
#plink_file_path = snp_calls_to_plink(
        #contig="3L:15,000,000-41,000,000", n_snps=10_000, min_minor_ac=5, thin_offset=0, max_missing_an=0
#    )

>>> building drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.10000.5.0.0
access SNP calls
count alleles
[########################################] | 100% Completed | 15m 2s
ascertain segregating biallelic sites
ascertained 1,419,515 sites
thin sites
thinned to 10,068 sites
set up dataset
compute genotype ref counts
[########################################] | 100% Completed | 479.38 s
ensure genotypes vary
final no. variants 10068
load final data
write plink files


In [ ]:
!ls -lh {results_dir}

total 0


### ADMIXTURE

In [10]:
!wget --no-clobber https://dalexander.github.io/admixture/binaries/admixture_linux-1.3.0.tar.gz

--2023-11-13 11:18:56--  https://dalexander.github.io/admixture/binaries/admixture_linux-1.3.0.tar.gz
Resolving dalexander.github.io (dalexander.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to dalexander.github.io (dalexander.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1916705 (1.8M) [application/gzip]
Saving to: ‘admixture_linux-1.3.0.tar.gz’

admixture_linux-1.3 100%[===================>]   1.83M  --.-KB/s    in 0.07s   

2023-11-13 11:18:57 (24.8 MB/s) - ‘admixture_linux-1.3.0.tar.gz’ saved [1916705/1916705]



In [11]:
!tar zxvf admixture_linux-1.3.0.tar.gz

dist/admixture_linux-1.3.0/
dist/admixture_linux-1.3.0/README.32.txt
dist/admixture_linux-1.3.0/admixture
dist/admixture_linux-1.3.0/admixture32
dist/admixture_linux-1.3.0/admixture-manual.pdf


In [12]:
!ln -s dist/admixture_linux-1.3.0/admixture

In [13]:
!./admixture

****                   ADMIXTURE Version 1.3.0                  ****
****                    Copyright 2008-2015                     ****
****           David Alexander, Suyash Shringarpure,            ****
****                John  Novembre, Ken Lange                   ****
****                                                            ****
****                 Please cite our paper!                     ****
****   Information at www.genetics.ucla.edu/software/admixture  ****

Usage: admixture <input file> <K>
See --help or manual for more advanced usage.


In [ ]:
!./admixture --help

****                   ADMIXTURE Version 1.3.0                  ****
****                    Copyright 2008-2015                     ****
****           David Alexander, Suyash Shringarpure,            ****
****                John  Novembre, Ken Lange                   ****
****                                                            ****
****                 Please cite our paper!                     ****
****   Information at www.genetics.ucla.edu/software/admixture  ****

                                                                              
  ADMIXTURE basic usage:  (see manual for complete reference)                 
    % admixture [options] inputFile K                                         
                                                                              
  where:                                                                      
    K is the number of populations; and                                       
    inputFile may be:                     

In [14]:
def run_admixture(
    contig,
    n_snps,
    thin_offset,
    min_minor_ac,
    K,
    sample_sets=None,
    sample_query=None,
    seed=42,
):

    plink_file_path = snp_calls_to_plink(
        contig=contig, n_snps=n_snps, min_minor_ac=min_minor_ac, thin_offset=thin_offset, max_missing_an=0, sample_sets = sample_sets, sample_query = sample_query
    )

    bed_file_path = f"{plink_file_path}.bed"
    fam_file_path = f"{plink_file_path}.fam"
    admixture_dir = f"{plink_file_path}.admixture/{seed}"
    !mkdir -pv {admixture_dir}
    Q_file_path = f"{admixture_dir}/{K}.Q"
    P_file_path = f"{admixture_dir}/{K}.P"
    log_file_path = f"{admixture_dir}/{K}.log"

    # run admixture if needed
    if not os.path.exists(Q_file_path):
        print(f"building {admixture_dir} K={K}")
        !./admixture -j2 --cv --seed={seed} {bed_file_path} {K} > {log_file_path}
        # move files to correct location
        stem = plink_file_path.split("/")[-1]
        !mv -v {stem}.{K}.Q {Q_file_path}
        !mv -v {stem}.{K}.P {P_file_path}

    # load results - sample IDs
    df_fam = pd.read_csv(
        fam_file_path,
        sep=" ",
        header=None,
        names=["family_id", "sample_id", "father", "mother", "sex", "phenotype"],
        index_col=False,
    )
    samples = df_fam["sample_id"]
    df_samples = ag3.sample_metadata(sample_sets=sample_sets, sample_query=sample_query)
    df_samples = (df_samples.set_index("sample_id").loc[samples]) #loc here to restrict to downsampled samples.

    # load results - ancestry fractions
    df_q = pd.read_csv(
        Q_file_path,
        sep=" ",
        header=None,
        names=[f"pop{i}" for i in range(K)],
        index_col=False,
    )
    df_q["popmax"] = df_q.idxmax(axis="columns")
    df_q["popmax_frac"] = df_q.apply(lambda row: row[row["popmax"]], axis="columns")
    df_q.set_index(samples, inplace=True)

    df_out = df_q.join(df_samples).reset_index()
    df_out.attrs["K"] = K
    return df_out


In [ ]:
#test
#for i in range(2, 4):
  #run_admixture(contig="3L:15,000,000-41,000,000", n_snps=1_000, min_minor_ac=5, thin_offset=0, sample_query='country == "Tanzania" or country == "Kenya"', K=i, seed=42)

>>> building drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.1000.5.0.0
access SNP calls


/usr/local/lib/python3.10/dist-packages/xarray/core/indexing.py:1443: PerformanceWarning: Slicing with an out-of-order index is generating 13 times more chunks
  return self.array[key]
/usr/local/lib/python3.10/dist-packages/xarray/core/indexing.py:1443: PerformanceWarning: Slicing with an out-of-order index is generating 16 times more chunks
  return self.array[key]


count alleles
[########################################] | 100% Completed | 361.37 s
ascertain segregating biallelic sites
ascertained 831,316 sites
thin sites
thinned to 1,000 sites
set up dataset
compute genotype ref counts
[########################################] | 100% Completed | 178.26 s
ensure genotypes vary
final no. variants 1001
load final data
write plink files
mkdir: created directory 'drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.1000.5.0.0.admixture'
mkdir: created directory 'drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.1000.5.0.0.admixture/42'
building drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.1000.5.0.0.admixture/42 K=2
copied 'chr3L:15.000.000-41.000.000.1000.5.0.0.2.Q' -> 'drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.1000.5.0.0.admixture/42/2.Q'
removed 'chr3L:15.000.000-41.000.000.1000.5.0.0.2.Q'
copied 'chr3L:15.000.000-41.000.000.1000.5.0.0.2.P' -> 'drive/MyD

In [15]:
df_samples = ag3.sample_metadata()

In [28]:
df_samples = ag3.sample_metadata(sample_query = 'taxon in ["gambiae","coluzzii"] and country == "Burkina Faso" or taxon in ["gambiae","coluzzii","arabiensis","gcx3"] and country == "Uganda" or country == "Tanzania" or country == "Kenya"')

In [29]:
df_samples.groupby(['country','taxon']).size()

country       taxon     
Burkina Faso  coluzzii      135
              gambiae       158
Kenya         arabiensis     13
              gambiae        19
              gcx3           54
Tanzania      arabiensis    225
              gambiae        64
              gcx3           11
Uganda        arabiensis     82
              gambiae       207
dtype: int64

In [32]:
#for seed in range(2):
for i in range(2, 7):
  run_admixture(contig="3L:15,000,000-41,000,000", n_snps=50_000, min_minor_ac=5, thin_offset=0, sample_query='taxon in ["gambiae","coluzzii"] and country == "Burkina Faso" or taxon in ["gambiae","coluzzii","arabiensis","gcx3"] and country == "Uganda" or country == "Tanzania" or country == "Kenya"', K=i, seed=42)

>>> building drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0
access SNP calls
count alleles
[########################################] | 100% Completed | 351.59 s
ascertain segregating biallelic sites
ascertained 1,514,823 sites
thin sites
thinned to 50,494 sites
set up dataset
compute genotype ref counts
[########################################] | 100% Completed | 236.80 s
ensure genotypes vary
final no. variants 50495
load final data
write plink files
mkdir: created directory 'drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture'
mkdir: created directory 'drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/42'
building drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/42 K=2
copied 'chr3L:15.000.000-41.000.000.50000.5.0.0.2.Q' -> 'drive/MyDrive/Tanzania/TZ-ref-admixture-131123/chr3L:15.000.000-41.000.000.50000.

In [ ]:
!ls -lh {results_dir}/*.admixture/*

'drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/0':
total 14M
-rw-------  1 root root  2.3K Nov  7 12:07 2.log
-rw-------+ 1 root root 1023K Nov  7 12:07 2.P
-rw-------+ 1 root root   360 Nov  7 12:07 2.Q
-rw-------  1 root root  2.7K Nov  7 12:07 3.log
-rw-------+ 1 root root  1.5M Nov  7 12:07 3.P
-rw-------+ 1 root root   540 Nov  7 12:07 3.Q
-rw-------  1 root root  3.2K Nov  7 12:07 4.log
-rw-------+ 1 root root  2.0M Nov  7 12:07 4.P
-rw-------+ 1 root root   720 Nov  7 12:07 4.Q
-rw-------  1 root root  3.6K Nov  7 12:08 5.log
-rw-------+ 1 root root  2.5M Nov  7 12:08 5.P
-rw-------+ 1 root root   900 Nov  7 12:08 5.Q
-rw-------  1 root root  4.0K Nov  7 12:09 6.log
-rw-------+ 1 root root  3.0M Nov  7 12:09 6.P
-rw-------+ 1 root root  1.1K Nov  7 12:09 6.Q
-rw-------  1 root root  3.5K Nov  7 12:10 7.log
-rw-------+ 1 root root  3.5M Nov  7 12:10 7.P
-rw-------+ 1 root root  1.3K Nov  7 12:10 7.Q

'drive/MyDrive/Tanzania/TZ-admixt

In [ ]:
!grep CV {results_dir}/*.admixture/*/*.log

drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.1000.5.0.0.admixture/42/2.log:CV error (K=2): 0.29653
drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.1000.5.0.0.admixture/42/3.log:CV error (K=3): 0.25874
drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/0/2.log:CV error (K=2): 0.33245
drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/0/3.log:CV error (K=3): 0.25258
drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/0/4.log:CV error (K=4): 0.25198
drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/0/5.log:CV error (K=5): 0.25281
drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/0/6.log:CV error (K=6): 0.26339
drive/MyDrive/Tanzania/TZ-admixture-021123/chr3L:15.000.000-41.000.000.50000.5.0.0.admixture/0/7.log:CV error (K=7): 0.26299


In [ ]:
######

In [ ]:
def plot_admixture(
    data_frame,
):

    data_frame = data_frame.sort_values(by=["popmax", "popmax_frac"], ascending=False)
    K = data_frame.attrs["K"]
    admx_cols = [f"pop{i}" for i in range(K)]

    fig = px.bar(
        data_frame=data_frame,
        x="sample_id",
        y=admx_cols,
        hover_data=["cohort", "PCA_cohort", "location", "year", "month", "season"],
        range_y=(0, 1),
        labels={
            "value": "Ancestry fraction",
            "sample_id": "Sample",
            "variable": "Ancestral population",
        },
        height=300,
    )

    fig.update_layout(
        title=f"K = {K}",
        xaxis=dict(
            tickmode='array',
            tickvals=[],
            ticktext=[]
        ),
        bargap=0,
    )

    fig.update_traces(
        marker=dict(line=dict(width=0)),
    )

    return fig

In [ ]:
df = run_admixture(contig="KB663610", n_snps=50_000, min_minor_ac=3, thin_offset=0, K=3)
plot_admixture(df)